In [2]:
import wrds
import pandas as pd
import numpy as np
import re
from pathlib import Path

Path("data").mkdir(exist_ok=True)
Path("data/raw").mkdir(exist_ok=True)
Path("data/processed").mkdir(exist_ok=True)

In [3]:
conn = wrds.Connection(wrds_username="sihan321")

Loading library list...
Done


In [4]:
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(response.text))

sp500 = tables[0]

sp500 = sp500.rename(columns={
    "Symbol": "ticker",
    "Security": "security",
    "GICS Sector": "sector",
    "GICS Sub-Industry": "sub_industry"
})

sp500["ticker_clean"] = (
    sp500["ticker"]
    .astype(str)
    .str.upper()
    .str.replace(".", "-", regex=False)
)

sp500[["ticker", "ticker_clean", "security", "sector", "sub_industry"]].head()

,ticker,ticker_clean,security,sector,sub_industry
0,MMM,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,AOS,A. O. Smith,Industrials,Building Products
2,ABT,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,ACN,Accenture,Information Technology,IT Consulting & Other Services


In [5]:
clean_calls_with_ticker = conn.raw_sql("""
WITH filtered AS (
    SELECT
        companyid,
        companyname,
        keydevid,
        transcriptid,
        headline,
        mostimportantdateutc,
        keydeveventtypename,
        transcriptcollectiontypename,
        transcriptpresentationtypename,
        transcriptcreationdate_utc
    FROM ciq_transcripts.wrds_transcript_detail
    WHERE keydeveventtypename = 'Earnings Calls'
      AND transcriptpresentationtypename = 'Final'
      AND mostimportantdateutc BETWEEN '2021-01-01' AND '2025-12-31'
      AND LOWER(headline) LIKE '%%earnings call%%'
      AND LOWER(headline) NOT LIKE '%%pre recorded%%'
      AND LOWER(headline) NOT LIKE '%%pre-recorded%%'
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY companyid, mostimportantdateutc
            ORDER BY
                CASE
                    WHEN transcriptcollectiontypename = 'Audited Copy' THEN 1
                    WHEN transcriptcollectiontypename = 'Proofed Copy' THEN 2
                    WHEN transcriptcollectiontypename = 'Edited Copy' THEN 3
                    WHEN transcriptcollectiontypename = 'EV Copy' THEN 4
                    ELSE 5
                END,
                transcriptcreationdate_utc DESC,
                transcriptid DESC
        ) AS rn
    FROM filtered
),
clean_calls AS (
    SELECT *
    FROM ranked
    WHERE rn = 1
),
ticker_matched AS (
    SELECT
        c.companyid,
        c.companyname,
        c.keydevid,
        c.transcriptid,
        c.headline,
        c.mostimportantdateutc AS call_date,
        c.transcriptcollectiontypename,
        t.ticker,
        t.startdate AS ticker_startdate,
        t.enddate AS ticker_enddate,
        t.primaryflag
    FROM clean_calls AS c
    LEFT JOIN ciq_common.wrds_ticker AS t
        ON c.companyid = t.companyid
       AND t.primaryflag = 1
       AND t.startdate <= c.mostimportantdateutc
       AND (t.enddate IS NULL OR t.enddate >= c.mostimportantdateutc)
)
SELECT *
FROM ticker_matched
WHERE ticker IS NOT NULL
ORDER BY companyname, call_date
""")

print(clean_calls_with_ticker.shape)
clean_calls_with_ticker.head()

(148172, 11)


,companyid,companyname,keydevid,transcriptid,headline,call_date,transcriptcollectiontypename,ticker,ticker_startdate,ticker_enddate,primaryflag
0,160130.0,01 Quantum Inc.,701176338.0,2249638.0,"01 Communique Laboratory Inc., Q4 2020 Earning...",2021-01-14,Audited Copy,ONE,2002-07-13,<NA>,1
1,160130.0,01 Quantum Inc.,717879688.0,2316497.0,"01 Communique Laboratory Inc., Q2 2021 Earning...",2021-06-10,Edited Copy,ONE,2002-07-13,<NA>,1
2,160130.0,01 Quantum Inc.,1680452476.0,2405134.0,"01 Communique Laboratory Inc., Q3 2021 Earning...",2021-09-09,Proofed Copy,ONE,2002-07-13,<NA>,1
3,160130.0,01 Quantum Inc.,1763820791.0,2485819.0,"01 Communique Laboratory Inc., 2021 Earnings C...",2022-01-13,Audited Copy,ONE,2002-07-13,<NA>,1
4,160130.0,01 Quantum Inc.,1774364602.0,2533379.0,"01 Communique Laboratory Inc., Q1 2022 Earning...",2022-03-17,Audited Copy,ONE,2002-07-13,<NA>,1


In [6]:
clean_calls_with_ticker.to_csv(
    "data/raw/clean_earnings_call_metadata_2021_2025.csv",
    index=False
)

In [7]:
clean_calls_with_ticker["ticker"] = clean_calls_with_ticker["ticker"].astype(str).str.upper()

clean_calls_sp500 = clean_calls_with_ticker.merge(
    sp500[["ticker_clean", "security", "sector", "sub_industry"]],
    left_on="ticker",
    right_on="ticker_clean",
    how="inner"
)

print(clean_calls_sp500.shape)
print(clean_calls_sp500["ticker"].nunique())

clean_calls_sp500.head()

(10766, 15)
493


,companyid,companyname,keydevid,transcriptid,headline,call_date,transcriptcollectiontypename,ticker,ticker_startdate,ticker_enddate,primaryflag,ticker_clean,security,sector,sub_industry
0,256329698.0,1&1 AG,704417895.0,2212548.0,"1&1 Drillisch AG, Q4 2020 Earnings Call, Feb 1...",2021-02-15,Edited Copy,DRI,2017-09-09,<NA>,1,DRI,Darden Restaurants,Consumer Discretionary,Restaurants
1,256329698.0,1&1 AG,709546971.0,2244374.0,"1&1 Drillisch AG, 2020 Earnings Call, Mar 25, ...",2021-03-25,Edited Copy,DRI,2017-09-09,<NA>,1,DRI,Darden Restaurants,Consumer Discretionary,Restaurants
2,256329698.0,1&1 AG,714259265.0,2279529.0,"1&1 Drillisch AG, Q1 2021 Earnings Call, May 1...",2021-05-11,Edited Copy,DRI,2017-09-09,<NA>,1,DRI,Darden Restaurants,Consumer Discretionary,Restaurants
3,256329698.0,1&1 AG,1676316704.0,2366516.0,"1&1 AG, H1 2021 Earnings Call, Aug 05, 2021",2021-08-05,Edited Copy,DRI,2017-09-09,<NA>,1,DRI,Darden Restaurants,Consumer Discretionary,Restaurants
4,256329698.0,1&1 AG,1757355285.0,2432056.0,"1&1 AG, Q3 2021 Earnings Call, Nov 09, 2021",2021-11-09,Edited Copy,DRI,2017-09-09,<NA>,1,DRI,Darden Restaurants,Consumer Discretionary,Restaurants


In [8]:
firm_call_count_sp500 = (
    clean_calls_sp500
    .groupby(["companyid", "companyname", "ticker", "security", "sector"])
    .size()
    .reset_index(name="n_calls")
)

firm_sample_sp500_200 = (
    firm_call_count_sp500
    .query("n_calls >= 12")
    .sort_values(["n_calls", "ticker"], ascending=[False, True])
    .head(200)
)

metadata_sample_sp500_200 = clean_calls_sp500[
    clean_calls_sp500["companyid"].isin(firm_sample_sp500_200["companyid"])
].copy()

metadata_sample_sp500_200.to_csv(
    "data/raw/metadata_sample_sp500_200_firms.csv",
    index=False
)

print(metadata_sample_sp500_200.shape)
print(metadata_sample_sp500_200["companyid"].nunique())

metadata_sample_sp500_200.head()

(4021, 15)
200


,companyid,companyname,keydevid,transcriptid,headline,call_date,transcriptcollectiontypename,ticker,ticker_startdate,ticker_enddate,primaryflag,ticker_clean,security,sector,sub_industry
0,256329698.0,1&1 AG,704417895.0,2212548.0,"1&1 Drillisch AG, Q4 2020 Earnings Call, Feb 1...",2021-02-15,Edited Copy,DRI,2017-09-09,<NA>,1,DRI,Darden Restaurants,Consumer Discretionary,Restaurants
1,256329698.0,1&1 AG,709546971.0,2244374.0,"1&1 Drillisch AG, 2020 Earnings Call, Mar 25, ...",2021-03-25,Edited Copy,DRI,2017-09-09,<NA>,1,DRI,Darden Restaurants,Consumer Discretionary,Restaurants
2,256329698.0,1&1 AG,714259265.0,2279529.0,"1&1 Drillisch AG, Q1 2021 Earnings Call, May 1...",2021-05-11,Edited Copy,DRI,2017-09-09,<NA>,1,DRI,Darden Restaurants,Consumer Discretionary,Restaurants
3,256329698.0,1&1 AG,1676316704.0,2366516.0,"1&1 AG, H1 2021 Earnings Call, Aug 05, 2021",2021-08-05,Edited Copy,DRI,2017-09-09,<NA>,1,DRI,Darden Restaurants,Consumer Discretionary,Restaurants
4,256329698.0,1&1 AG,1757355285.0,2432056.0,"1&1 AG, Q3 2021 Earnings Call, Nov 09, 2021",2021-11-09,Edited Copy,DRI,2017-09-09,<NA>,1,DRI,Darden Restaurants,Consumer Discretionary,Restaurants


In [9]:
component_desc = conn.describe_table(
    library="ciq_transcripts",
    table="ciqtranscriptcomponent"
)

component_desc[["name", "type"]]

Approximately 91226539 rows in ciq_transcripts.ciqtranscriptcomponent.


,name,type
0,transcriptcomponentid,INTEGER
1,transcriptid,INTEGER
2,componentorder,SMALLINT
3,transcriptcomponenttypeid,SMALLINT
4,transcriptpersonid,INTEGER
5,componenttext,VARCHAR


In [10]:
transcript_ids = (
    metadata_sample_sp500_200["transcriptid"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

len(transcript_ids)

4021

In [16]:
try:
    conn.connection.rollback()
    print("Rollback successful.")
except Exception as e:
    print("Rollback failed:", e)

Rollback successful.


In [17]:
from pathlib import Path

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

Path("data/raw/transcript_batches").mkdir(parents=True, exist_ok=True)

batch_size = 300

for batch_no, ids in enumerate(chunks(transcript_ids, batch_size), start=1):
    ids_str = ",".join(map(str, ids))
    
    components = conn.raw_sql(f"""
    SELECT
        transcriptid,
        transcriptcomponentid,
        componentorder,
        componenttext
    FROM ciq_transcripts.ciqtranscriptcomponent
    WHERE transcriptid IN ({ids_str})
    ORDER BY transcriptid, componentorder
    """)
    
    full_texts = (
        components
        .dropna(subset=["componenttext"])
        .sort_values(["transcriptid", "componentorder"])
        .groupby("transcriptid")["componenttext"]
        .apply(lambda x: "\n".join(x.astype(str)))
        .reset_index()
        .rename(columns={"componenttext": "transcript_text"})
    )
    
    out_path = f"data/raw/transcript_batches/full_text_batch_{batch_no:03d}.csv"
    full_texts.to_csv(out_path, index=False)
    
    print(f"Batch {batch_no}: {full_texts.shape}")

Batch 1: (300, 2)
Batch 2: (300, 2)
Batch 3: (300, 2)
Batch 4: (300, 2)
Batch 5: (300, 2)
Batch 6: (300, 2)
Batch 7: (300, 2)
Batch 8: (300, 2)
Batch 9: (300, 2)
Batch 10: (300, 2)
Batch 11: (300, 2)
Batch 12: (300, 2)
Batch 13: (300, 2)
Batch 14: (121, 2)


In [18]:
files = sorted(Path("data/raw/transcript_batches").glob("full_text_batch_*.csv"))

full_texts = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)

dataset_text = metadata_sample_sp500_200.merge(
    full_texts,
    on="transcriptid",
    how="left"
)

dataset_text.to_csv(
    "data/raw/earnings_call_transcripts_sp500_200_with_text.csv",
    index=False
)

print(dataset_text.shape)
print(dataset_text["transcript_text"].notna().mean())
dataset_text.head()

(4021, 16)
1.0


,companyid,companyname,keydevid,transcriptid,headline,call_date,transcriptcollectiontypename,ticker,ticker_startdate,ticker_enddate,primaryflag,ticker_clean,security,sector,sub_industry,transcript_text
0,256329698.0,1&1 AG,704417895.0,2212548.0,"1&1 Drillisch AG, Q4 2020 Earnings Call, Feb 1...",2021-02-15,Edited Copy,DRI,2017-09-09,<NA>,1,DRI,Darden Restaurants,Consumer Discretionary,Restaurants,[Interpreted] I'm delighted to welcome you her...
1,256329698.0,1&1 AG,709546971.0,2244374.0,"1&1 Drillisch AG, 2020 Earnings Call, Mar 25, ...",2021-03-25,Edited Copy,DRI,2017-09-09,<NA>,1,DRI,Darden Restaurants,Consumer Discretionary,Restaurants,"Thank you, operator. Welcome, ladies and gentl..."
2,256329698.0,1&1 AG,714259265.0,2279529.0,"1&1 Drillisch AG, Q1 2021 Earnings Call, May 1...",2021-05-11,Edited Copy,DRI,2017-09-09,<NA>,1,DRI,Darden Restaurants,Consumer Discretionary,Restaurants,"Good day, and welcome to the 1&1 Drillisch AG ..."
3,256329698.0,1&1 AG,1676316704.0,2366516.0,"1&1 AG, H1 2021 Earnings Call, Aug 05, 2021",2021-08-05,Edited Copy,DRI,2017-09-09,<NA>,1,DRI,Darden Restaurants,Consumer Discretionary,Restaurants,"Welcome, ladies and gentlemen. We welcome you ..."
4,256329698.0,1&1 AG,1757355285.0,2432056.0,"1&1 AG, Q3 2021 Earnings Call, Nov 09, 2021",2021-11-09,Edited Copy,DRI,2017-09-09,<NA>,1,DRI,Darden Restaurants,Consumer Discretionary,Restaurants,"Dear ladies and gentlemen, welcome to the 1&1 ..."


In [19]:
from pathlib import Path

Path("data/processed").mkdir(parents=True, exist_ok=True)

dataset_text.to_csv(
    "data/processed/earnings_call_transcripts_sp500_200_with_text.csv",
    index=False
)

In [20]:
summary = {
    "n_rows": dataset_text.shape[0],
    "n_columns": dataset_text.shape[1],
    "n_companies": dataset_text["companyid"].nunique(),
    "n_tickers": dataset_text["ticker"].nunique(),
    "n_transcripts": dataset_text["transcriptid"].nunique(),
    "text_match_rate": dataset_text["transcript_text"].notna().mean(),
    "start_date": dataset_text["call_date"].min(),
    "end_date": dataset_text["call_date"].max()
}

summary

{'n_rows': 4021,
 'n_columns': 16,
 'n_companies': 200,
 'n_tickers': 189,
 'n_transcripts': 4021,
 'text_match_rate': np.float64(1.0),
 'start_date': '2021-01-07',
 'end_date': '2025-12-19'}

In [21]:
from pathlib import Path
import pandas as pd

Path("data/processed").mkdir(parents=True, exist_ok=True)

pd.DataFrame([summary]).to_csv(
    "data/processed/sample_summary_sp500_200.csv",
    index=False
)

In [25]:
import re
import numpy as np
import pandas as pd
ai_keywords = [
    "ai",
    "artificial intelligence",
    "generative ai",
    "gen ai",
    "genai",
    "machine learning",
    "deep learning",
    "natural language processing",
    "large language model",
    "large language models",
    "llm",
    "llms",
    "foundation model",
    "foundation models",
    "neural network",
    "neural networks",
    "computer vision",
    "predictive ai",
    "ai model",
    "ai models",
    "ai platform",
    "ai platforms",
    "ai assistant",
    "ai assistants",
    "chatbot",
    "chatbots",
    "chatgpt",
    "gpt",
    "gpt-3",
    "gpt-4",
    "openai",
    "copilot",
    "gemini",
    "claude",
    "anthropic"
]

ai_pattern = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in ai_keywords) + r")\b",
    flags=re.IGNORECASE
)

def split_sentences(text):
    if pd.isna(text):
        return []
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", str(text)) if s.strip()]

def compute_ai_vars(text):
    sentences = split_sentences(text)
    total = len(sentences)
    ai_count = sum(1 for s in sentences if ai_pattern.search(s))
    
    return pd.Series({
        "total_sentences": total,
        "ai_sentence_count": ai_count,
        "ai_intensity": ai_count / total if total > 0 else np.nan
    })

ai_vars = dataset_text["transcript_text"].apply(compute_ai_vars)

dataset_processed = pd.concat(
    [dataset_text.drop(columns=["transcript_text"]), ai_vars],
    axis=1
)

In [30]:
from pathlib import Path

Path("data/processed").mkdir(parents=True, exist_ok=True)

dataset_processed.to_csv(
    "data/processed/earnings_call_ai_intensity_sp500_200.csv",
    index=False
)

In [26]:
dataset_processed[
    ["ticker", "call_date", "headline", "total_sentences", "ai_sentence_count", "ai_intensity"]
].head(20)

,ticker,call_date,headline,total_sentences,ai_sentence_count,ai_intensity
0,DRI,2021-02-15,"1&1 Drillisch AG, Q4 2020 Earnings Call, Feb 1...",527.0,0.0,0.000000
1,DRI,2021-03-25,"1&1 Drillisch AG, 2020 Earnings Call, Mar 25, ...",400.0,0.0,0.000000
2,DRI,2021-05-11,"1&1 Drillisch AG, Q1 2021 Earnings Call, May 1...",329.0,0.0,0.000000
3,DRI,2021-08-05,"1&1 AG, H1 2021 Earnings Call, Aug 05, 2021",515.0,0.0,0.000000
4,DRI,2021-11-09,"1&1 AG, Q3 2021 Earnings Call, Nov 09, 2021",408.0,0.0,0.000000
5,DRI,2022-03-18,"1&1 AG, 2021 Earnings Call, Mar 18, 2022",559.0,0.0,0.000000
6,DRI,2022-05-12,"1&1 AG, Q1 2022 Earnings Call, May 12, 2022",397.0,0.0,0.000000
7,DRI,2022-08-04,"1&1 AG, H1 2022 Earnings Call, Aug 04, 2022",382.0,0.0,0.000000
8,DRI,2022-11-10,"1&1 AG, Q3 2022 Earnings Call, Nov 10, 2022",358.0,0.0,0.000000
9,DRI,2023-03-30,"1&1 AG, 2022 Earnings Call, Mar 30, 2023",500.0,0.0,0.000000


In [27]:
dataset_processed[["total_sentences", "ai_sentence_count", "ai_intensity"]].describe()

,total_sentences,ai_sentence_count,ai_intensity
count,4021.00000,4021.000000,4021.000000
mean,484.54290,3.768217,0.007684
std,111.91686,10.920267,0.022174
min,82.00000,0.000000,0.000000
25%,425.00000,0.000000,0.000000
50%,484.00000,0.000000,0.000000
75%,545.00000,2.000000,0.003344
max,1108.00000,117.000000,0.211957


In [28]:
(dataset_processed["ai_sentence_count"] > 0).mean()

np.float64(0.367072867445909)

In [29]:
def extract_ai_sentences(text):
    sentences = split_sentences(text)
    return " ||| ".join([s for s in sentences if ai_pattern.search(s)])

dataset_processed["ai_sentences"] = dataset_text["transcript_text"].apply(extract_ai_sentences)

dataset_processed.loc[
    dataset_processed["ai_sentence_count"] > 0,
    ["ticker", "call_date", "headline", "ai_sentence_count", "ai_intensity", "ai_sentences"]
].head(20)

,ticker,call_date,headline,ai_sentence_count,ai_intensity,ai_sentences
13,DRI,2024-03-21,"1&1 AG, 2023 Earnings Call, Mar 21, 2024",2.0,0.002829,"Now when I look at what's happening with AI, t..."
19,DRI,2025-08-07,"1&1 AG, Q2 2025 Earnings Call, Aug 07, 2025",1.0,0.002457,"Additionally, we can invest it in AI."
31,ABT,2023-07-20,"Abbott Laboratories, Q2 2023 Earnings Call, Ju...",2.0,0.003839,And if I think about everything that's going o...
40,ABT,2025-10-15,"Abbott Laboratories, Q3 2025 Earnings Call, Oc...",1.0,0.001715,"This quarter, we actually bought an AI-powered..."
43,ABBV,2021-07-30,"AbbVie Inc., Q2 2021 Earnings Call, Jul 30, 2021",1.0,0.001481,"And lastly, in eye care, at the recent meeting..."
58,ABBV,2025-04-25,"AbbVie Inc., Q1 2025 Earnings Call, Apr 25, 2025",1.0,0.001418,The team has applied machine learning to these...
59,ABBV,2025-07-31,"AbbVie Inc., Q2 2025 Earnings Call, Jul 31, 2025",1.0,0.001792,"So again, pretty important in terms of how we ..."
61,ACN,2021-03-18,"Accenture plc, Q2 2021 Earnings Call, Mar 18, ...",6.0,0.013605,COVID has hit a giant fast-forward button to t...
62,ACN,2021-06-24,"Accenture plc, Q3 2021 Earnings Call, Jun 24, ...",3.0,0.007143,SynOps will deliver AI-powered insights and hi...
63,ACN,2021-09-23,"Accenture plc, Q4 2021 Earnings Call, Sep 23, ...",2.0,0.004098,"Just over 1 year ago, we created Accenture Clo..."
